# GOBR — execução no Google Colab (GPU)

Este notebook **não substitui o projeto no Windows**. Ele baixa a mesma codebase do GitHub, instala as dependências, usa o Ollama **dentro da máquina do Colab** e executa a primeira entrega e a geração incremental de explicações (parser → interpretador → explicador).

**O que fica onde**
- **Windows:** edição da codebase e commits.
- **GitHub:** versão oficial do código para a disciplina.
- **Colab:** execução temporária com GPU (quando disponível).
- **Drive:** resultados das explicações.

O Colab gratuito tem limites variáveis de GPU/tempo e pode desconectar. **Não é um servidor permanente**. O Ollama não será exposto à internet.

**Antes de executar:** envie a pasta do projeto para um repositório no GitHub. Use um repositório público para o comando de clone mostrado aqui. Para repositório privado, não coloque tokens no código do notebook; use a alternativa de upload de ZIP ou configure autenticação de maneira segura.

## 1. Ative a GPU
No menu do Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: T4 GPU** (ou outra GPU disponível). Depois clique no botão ▶ desta célula.

A GPU deve aparecer em `nvidia-smi`. Se não aparecer, volte às configurações do ambiente e escolha uma GPU.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

## 2. Monte o Google Drive
O botão ▶ abrirá uma janela pedindo permissão para o Colab acessar seu Drive. As saídas serão salvas em `Meu Drive/gobr-agent/outputs/`. O código fica no GitHub, não no Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Baixe a mesma codebase do GitHub
Substitua a URL abaixo pela URL **real** do seu repositório público no GitHub. Exemplo: `https://github.com/usuario/sistemas-multiagentes.git`.

Sempre que você alterar o projeto no Windows e executar `git push`, execute novamente esta célula no Colab para baixar a nova versão. **Não edite agentes no Colab**, pois as alterações são temporárias e não voltam automaticamente ao GitHub.

**Alternativa:** se ainda não publicou seu repositório, altere `ORIGEM` para `"zip"`; o Colab solicitará o upload de um ZIP do projeto, sem enviar tokens de acesso.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

ORIGEM = "github"  # "github" ou "zip"
REPO_URL = "https://github.com/gsilverio/sistemas-multiagentes.git"
BRANCH = "etapa-0-reestruturacao"  # branch com o codigo a executar ("main" depois do merge)
PROJETO = Path('/content/gobr-project')

if any(nome == 'src' or nome.startswith('src.') for nome in sys.modules):
    raise RuntimeError(
        'O código antigo ainda está carregado na memória. '
        'Reinicie a sessão do Colab antes de clonar a nova revisão.'
    )

if PROJETO.exists():
    shutil.rmtree(PROJETO)  # Apenas a copia temporaria do Colab

if ORIGEM == "github":
    if "SEU_USUARIO" in REPO_URL or "SEU_REPOSITORIO" in REPO_URL:
        raise ValueError("Troque REPO_URL pela URL real do seu repositorio GitHub.")
    if not REPO_URL.startswith("https://github.com/"):
        raise ValueError("Use uma URL HTTPS de repositorio github.com.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJETO)], check=True)
elif ORIGEM == "zip":
    from google.colab import files
    enviados = files.upload()
    if len(enviados) != 1:
        raise ValueError("Envie somente um arquivo ZIP da codebase.")
    nome_zip = next(iter(enviados))
    PROJETO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(nome_zip) as pacote:
        for member in pacote.infolist():
            destino = (PROJETO / member.filename).resolve()
            if not destino.is_relative_to(PROJETO.resolve()):
                raise ValueError(f"Caminho inseguro no ZIP: {member.filename}")
        pacote.extractall(PROJETO)
else:
    raise ValueError("ORIGEM deve ser 'github' ou 'zip'.")

if not (PROJETO / 'src' / 'gobr' / 'parser.py').is_file():
    raise FileNotFoundError("O projeto deve conter src/gobr/parser.py.")

os.chdir(PROJETO)
sys.path.insert(0, str(PROJETO))
print('Projeto carregado em:', PROJETO)
print('Arquivos de regras:', len(list((PROJETO / 'data' / 'referencias').glob('*.gobr'))))

## 4. Instale as dependências Python
Lê o mesmo `requirements.txt` versionado no GitHub. Esta instalação precisa ser refeita quando o Colab criar uma nova máquina virtual.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJETO / 'requirements.txt')],
    check=True
)
print('Dependencias instaladas.')

## 5. Instale e inicie o Ollama **somente no Colab**
O Ollama executa o modelo dentro da máquina virtual do Colab. O servidor estará acessível **apenas pelo próprio notebook** em `127.0.0.1:11434`, não pelo Windows.

O script de instalação abaixo é o procedimento fornecido no site oficial do Ollama.

In [ ]:
import shutil

if shutil.which('ollama'):
    print('Ollama já está instalado:', shutil.which('ollama'))
else:
    subprocess.run([
        'bash', '-lc',
        'set -euo pipefail; ' 
        'if ! command -v zstd >/dev/null 2>&1; then apt-get update -qq; apt-get install -y -qq zstd; fi; ' 
        'curl -fsSL https://ollama.com/install.sh -o /tmp/ollama-install.sh; ' 
        'bash /tmp/ollama-install.sh',
    ], check=True)
    print('Ollama instalado.')

In [ ]:
import time
import urllib.request

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'

def ollama_pronto():
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not ollama_pronto():
    log = open('/tmp/ollama-gobr.log', 'a')
    processo_ollama = subprocess.Popen(
        ['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT,
        env=os.environ.copy()
    )

for _ in range(45):
    if ollama_pronto():
        break
    time.sleep(1)
else:
    raise RuntimeError('Ollama nao iniciou. Consulte /tmp/ollama-gobr.log')

print('Ollama pronto em http://127.0.0.1:11434')

## 6. Baixe o modelo maior
Começaremos com `gemma3:12b` (~8,2 GB de download de modelo). Ele deverá caber em muitas GPUs T4 de 16 GB, mas o uso real depende do contexto e da memória disponível. **Baixar o modelo não garante que a inferência use inteiramente a GPU.**

Se houver erro de memória, podemos testar um modelo menor. O modelo pode ser alterado aqui sem modificar os agentes.

In [ ]:
MODELO = 'gemma3:12b'
CONTEXTO = 4096

processo_modelo = subprocess.Popen(
    ['ollama', 'pull', MODELO],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for linha in processo_modelo.stdout:
    print(linha, end='', flush=True)
codigo = processo_modelo.wait()
if codigo != 0:
    raise subprocess.CalledProcessError(codigo, ['ollama', 'pull', MODELO])
os.environ['GOBR_LLM_MODEL'] = MODELO
os.environ['GOBR_LLM_NUM_CTX'] = str(CONTEXTO)
os.environ['OLLAMA_BASE_URL'] = 'http://127.0.0.1:11434'
print('Modelo preparado:', MODELO)

## 7. Confira se o Ollama realmente usou a GPU
O comando abaixo mostra os modelos carregados e o dispositivo usado. Se aparecer processamento misto GPU/CPU, pode significar memória de vídeo insuficiente para manter todo o modelo na GPU.

In [ ]:
subprocess.run(['ollama', 'ps'], check=True)

## Quando você voltar amanhã
1. Se o código no GitHub mudou, reinicie a sessão do Colab (menu **Ambiente de execução → Reiniciar sessão**) antes de clonar, para evitar módulos Python antigos em memória.
2. Abra a versão atualizada deste notebook no GitHub.
3. Se necessário, ative a GPU e reconecte o Drive.
4. Execute as células de cima para baixo: a máquina virtual anterior pode ter sido apagada.
5. Os arquivos salvos no Drive continuam lá. O código original continua no GitHub.

Referências oficiais: [Google Colab — FAQ](https://research.google.com/colaboratory/intl/pt-BR/faq.html) e [Ollama — instalação Linux](https://ollama.com/download/linux).

## 8. Preparar as 1.700 regras extraídas

Se você acabou de abrir o notebook nesta sessão, execute antes as **etapas 1 a 6** (GPU, Drive, GitHub, dependências, Ollama e modelo).

A pasta de saída no Drive leva o nome do modelo e da **versão do explicador** (um hash dos arquivos que mudam o texto: glossário, rascunho, guard, prompt e agente). Commits que não mexem no explicador continuam na mesma pasta, e a geração retoma de onde parou.

In [ ]:
import sys
import subprocess
from pathlib import Path

if not (PROJETO / 'src' / 'agents' / 'explicador.py').is_file():
    raise FileNotFoundError('Atualize o repositório no GitHub, reinicie a sessão do Colab e clone novamente.')

subprocess.run([sys.executable, '-m', 'src.pipelines.prepare_references'], cwd=str(PROJETO), check=True)
ENTRADA_REGRAS = PROJETO / 'outputs' / 'generated' / 'regras_extraidas.jsonl'
assert ENTRADA_REGRAS.is_file()

VERSAO_EXPLICADOR = subprocess.run(
    [sys.executable, '-c', 'from src.pipelines.generate_explanations import versao_explicador; print(versao_explicador())'],
    cwd=str(PROJETO), capture_output=True, text=True, check=True,
).stdout.strip()
NOME_MODELO = MODELO.replace(':', '_').replace('/', '_')
DESTINO_LOTE = Path('/content/drive/MyDrive/gobr-agent/outputs/explicacoes') / f'{NOME_MODELO}_{VERSAO_EXPLICADOR}'
DESTINO_LOTE.mkdir(parents=True, exist_ok=True)
print('Entrada:', ENTRADA_REGRAS)
print('Saída permanente no Drive:', DESTINO_LOTE)

## 9. Prévia do lote (sem executar o modelo)

Mostra os IDs e nomes das próximas regras. Não consome GPU.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.pipelines.generate_explanations',
    '--input', str(ENTRADA_REGRAS),
    '--output-dir', str(DESTINO_LOTE),
    '--limit', '3', '--dry-run',
], cwd=str(PROJETO), check=True)

## 10. Primeiro lote: cinco explicações

Para cada regra o explicador monta um **rascunho determinístico** (sem LLM), faz **uma chamada** ao modelo para reescrever o texto e confere o resultado com o **guard**. Se o guard reprovar, faz uma chamada de correção com os problemas apontados; se reprovar de novo, usa o rascunho. Cada regra vira `items/<id>.json` no Drive assim que termina.

O método de cada item: `llm` (aprovado de primeira), `llm_corrigido` (aprovado na correção) ou `rascunho` (o modelo errou duas vezes). Blocos iguais a um já explicado aparecem como `reaproveitada de ...`.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.pipelines.generate_explanations',
    '--input', str(ENTRADA_REGRAS),
    '--output-dir', str(DESTINO_LOTE),
    '--limit', '5',
], cwd=str(PROJETO), check=True)

## 11. Examinar os resultados

Mostra o GOBR, o texto final, o método e, quando o modelo foi reprovado, o que o guard apontou.

In [ ]:
import json

for arquivo in sorted((DESTINO_LOTE / 'items').glob('*.json'))[:8]:
    item = json.loads(arquivo.read_text(encoding='utf-8'))
    explicacao = item.get('explicacao') or {}
    print('\n---', item['id'], '|', item['status'], '|', explicacao.get('metodo', '-'),
          '| aprovada' if explicacao.get('aprovada') else '')
    print(item['gobr'])
    print('>>', explicacao.get('texto') or item.get('reason') or str(item.get('error', ''))[:400])
    if explicacao.get('metodo') != 'llm' and explicacao.get('tentativas_llm'):
        print('   última tentativa do modelo:', explicacao['tentativas_llm'][-1][:300])
        print('   problemas:', explicacao.get('problemas'))

## 12. Gerar o restante (retomada automática)

Ative `EXECUTAR_MAIS = True` depois de conferir o primeiro lote. A célula processa `LOTE` regras novas e **pula as já concluídas**; execute de novo para o próximo lote. Com `gemma3:12b` numa T4, conte com 5 a 10 s por regra. Blocos repetidos (só o nome muda) reaproveitam a explicação sem chamar o modelo: são cerca de 1.200 chamadas, ≈ 2 a 3 horas no total. Se o Colab desconectar, nada se perde: refaça as etapas 1 a 8 e rode esta célula de novo.

`LOTE = 0` processa tudo o que falta numa só execução.

In [ ]:
EXECUTAR_MAIS = False
LOTE = 200

if EXECUTAR_MAIS:
    subprocess.run([
        sys.executable, '-m', 'src.pipelines.generate_explanations',
        '--input', str(ENTRADA_REGRAS),
        '--output-dir', str(DESTINO_LOTE),
        '--limit', str(LOTE),
    ], cwd=str(PROJETO), check=True)
else:
    print('Desativado. Altere EXECUTAR_MAIS para True depois de revisar o primeiro lote.')

## 12b. Avaliar as explicações e comparar com a linha de base

`explicacoes.jsonl` (no Drive, dentro de `DESTINO_LOTE`) é atualizado ao fim de cada lote. Esta célula confere todas as explicações com o guard e compara com as 59 explicações do pipeline antigo.

**Quando terminar as 1.700**, baixe `explicacoes.jsonl` do Drive e coloque no projeto em `data/explicacoes/explicacoes_v2.jsonl`: ele alimenta o RAG (etapa 3) e o fine-tuning (etapa 6).

In [ ]:
for arquivo in [DESTINO_LOTE / 'explicacoes.jsonl', PROJETO / 'data' / 'avaliacao' / 'explicacoes_baseline_v1.jsonl']:
    print('=' * 20, arquivo.name)
    subprocess.run([sys.executable, '-m', 'src.pipelines.avaliar_explicacoes', str(arquivo), '--exemplos', '3'],
                   cwd=str(PROJETO), check=True)
resumo = json.loads((DESTINO_LOTE / 'batch_summary.json').read_text(encoding='utf-8'))
print('Último lote:', resumo['metodos_nesta_execucao'], '| falhas:', resumo['falhas_nesta_execucao'])

## 13. Indexar as explicações no Drive

Depois de gerar e revisar pelo menos um lote, esta etapa cria um índice FAISS persistente usando embeddings multilíngues. Para fechar a entrega rapidamente, `--include-pending` inclui os itens concluídos ainda marcados como pendentes e grava essa condição no `manifest.json`; depois podemos reconstruir o índice apenas com itens aprovados.

In [ ]:
INDEX_DIR = DESTINO_LOTE / 'index_explanations'
ITEMS_DIR = DESTINO_LOTE / 'items'
if not any(ITEMS_DIR.glob('*.json')):
    raise FileNotFoundError('Gere pelo menos um lote de explicações antes de indexar.')
subprocess.run([
    sys.executable, '-m', 'src.rag.explanation_index',
    '--items-dir', str(ITEMS_DIR),
    '--output-dir', str(INDEX_DIR),
    '--include-pending',
], cwd=str(PROJETO), check=True)
print('Índice salvo em:', INDEX_DIR)

## 14. Testar recuperação sem chamar o Gemma

O teste abaixo confirma que o índice retorna regras semanticamente próximas. O índice e os metadados permanecem no Drive para serem reutilizados em uma etapa RAG posterior.

In [ ]:
from src.rag.explanation_index import search_index

for consulta in [
    'clientes sem venda devem ser visitados',
    'não agendar clientes inadimplentes',
    'frequência de atendimento para prospects',
]:
    print('\nCONSULTA:', consulta)
    for resultado in search_index(INDEX_DIR, consulta, k=3):
        print(round(resultado['score'], 3), resultado['id'], resultado['text'][:220])

## 15. Indexar o manual GOBR para apoio interno do agente

O manual fica em um índice separado das explicações. Nesta entrega, ele será usado como contexto interno para orientar a geração da regra GOBR; o texto recuperado não é exibido automaticamente ao usuário. Cada trecho mantém a página de origem para permitir uma rota de respostas diretas no futuro.

In [ ]:
MANUAL_PDF = PROJETO / 'data' / 'manual' / 'govendas-rules.pdf'
MANUAL_INDEX_DIR = Path('/content/drive/MyDrive/gobr-agent/outputs/manual_index')
if not MANUAL_PDF.is_file():
    raise FileNotFoundError(f'Manual não encontrado: {MANUAL_PDF}')
subprocess.run([
    sys.executable, '-m', 'src.rag.manual_index',
    '--pdf', str(MANUAL_PDF),
    '--output-dir', str(MANUAL_INDEX_DIR),
], cwd=str(PROJETO), check=True)
print('Índice do manual salvo em:', MANUAL_INDEX_DIR)

## 16. Testar a recuperação do manual sem chamar o Gemma

O teste confirma que consultas sobre conceitos do GOBR recuperam trechos e páginas do PDF. A consulta será ligada ao agente na etapa seguinte.

In [ ]:
from src.rag.manual_index import search_index as search_manual_index

for consulta in [
    'o que significa frequência de atendimento',
    'como declarar não agendar',
    'filtros temporais permitidos',
]:
    print('\nCONSULTA:', consulta)
    for resultado in search_manual_index(MANUAL_INDEX_DIR, consulta, k=3):
        meta = resultado['metadata']
        print(round(resultado['score'], 3), 'página', meta['pagina'], resultado['text'][:300])

### Ao retomar outro dia

1. Abra **este mesmo notebook**, ative a GPU, monte o Drive, clone o GitHub e inicie o Ollama (etapas 1 a 6).
2. Rode a etapa **8**: se o explicador não mudou, `DESTINO_LOTE` aponta para a mesma pasta do Drive.
3. Rode a etapa **12** com `EXECUTAR_MAIS = True` até terminar e confira com a **12b**.
4. Os índices (etapas seguintes) podem ser refeitos a qualquer momento a partir da mesma pasta.

Os JSONs salvos no Drive não somem com a desconexão. Itens `failed` são tentados de novo até o limite; `blocked` são regras com trechos que o glossário não traduz (as duas anomalias do acervo).